# Day 12 — Dataclasses

> ⚠️ **Why this matters.** Yesterday your `Word` class was 40+ lines of boilerplate (`__init__`, `__repr__`, `__eq__`, `__hash__`). With `@dataclass` it's 5 lines. Same behavior, less code, fewer bugs. **Every Python codebase you'll see from here on uses dataclasses.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/12-dataclasses.ipynb)

## What you'll do today

**Time:** 75 min lesson + 60 min refactor + 45 min quiz.

- [ ] You can convert any data-holding class to `@dataclass`
- [ ] You know `field(default_factory=...)` for mutable defaults
- [ ] You can use `frozen=True` for immutable objects
- [ ] Your `Word` class is now 5 lines

## 1. The transformation

In [ ]:
# Before (Day 11)
class Word:
    def __init__(self, word: str, ipa: str = '', thai: str = ''):
        self.word = word
        self.ipa = ipa
        self.thai = thai
    def __repr__(self):
        return f'Word({self.word!r}, ipa={self.ipa!r}, thai={self.thai!r})'
    def __eq__(self, other):
        if not isinstance(other, Word): return NotImplemented
        return (self.word, self.ipa, self.thai) == (other.word, other.ipa, other.thai)
    def __hash__(self):
        return hash((self.word, self.ipa, self.thai))

In [ ]:
# After (Day 12)
from dataclasses import dataclass

@dataclass(frozen=True)
class Word:
    word: str
    ipa: str = ''
    thai: str = ''

# That's it. __init__, __repr__, __eq__, __hash__ are generated.

**Test:**

In [ ]:
w1 = Word('thorough', '/ˈθʌrə/', 'ละเอียด')
w2 = Word('thorough', '/ˈθʌrə/', 'ละเอียด')
print(w1)             # Word(word='thorough', ipa='/ˈθʌrə/', thai='ละเอียด')
print(w1 == w2)       # True
print(hash(w1))       # int


## 2. Useful `@dataclass` options

| Option | Default | What |
|--------|---------|------|
| `frozen=True` | False | Instances immutable; you can hash them and use as dict keys |
| `order=True` | False | Generates `<`, `<=`, `>`, `>=` (uses tuple of fields) |
| `slots=True` | False | Saves memory, prevents adding new attrs (Python 3.10+) |
| `kw_only=True` | False | All fields must be passed by keyword |
| `eq=False` | True | Disable `__eq__` generation |

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Score:
    points: int
    name: str

scores = [Score(85, 'Alice'), Score(95, 'Bob'), Score(70, 'Carol')]
print(sorted(scores))   # uses tuple order: points first, then name

## 3. Mutable defaults — use `field(default_factory=...)`

In [ ]:
# WRONG:
# @dataclass
# class C:
#     items: list = []   # ValueError: mutable default

from dataclasses import dataclass, field

@dataclass
class WordStore:
    items: dict = field(default_factory=dict)
    tags: set = field(default_factory=set)

s1 = WordStore()
s2 = WordStore()
s1.items['x'] = 1
print(s2.items)   # {} — not affected, each has its own dict

## 4. Methods are still allowed

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Word:
    word: str
    ipa: str = ''

    def quiz_me(self, guess: str) -> bool:
        return guess.strip() == self.ipa

    def to_dict(self) -> dict:
        return {'word': self.word, 'ipa': self.ipa}

    @classmethod
    def from_dict(cls, d: dict) -> 'Word':
        return cls(word=d['word'], ipa=d.get('ipa', ''))

w = Word('thorough', '/ˈθʌrə/')
print(w.quiz_me('/ˈθʌrə/'))

## End-of-day mini-project

> 🎯 **Refactor `Word` and `WordEntry` everywhere to use `@dataclass`.**

### Steps

1. Rewrite `src/english_helper/word.py` using `@dataclass(frozen=True)`.
2. Update `storage.py` — replaces dict-based store with `dict[str, Word]`.
3. Update `cli.py` to construct `Word` instances.
4. Use `Word.to_dict()` / `Word.from_dict()` for JSON serialization.
5. mypy still passes.

### Verify

```python
from english_helper.word import Word
w = Word('thorough', ipa='/ˈθʌrə/', thai='ละเอียด')
assert w == Word('thorough', ipa='/ˈθʌrə/', thai='ละเอียด')
assert hash(w)
assert Word.from_dict(w.to_dict()) == w
```

## Connect to the project

> 🎯 **Tomorrow (Day 13):** properties, computed attributes, inheritance — for when one class isn't enough.

**Quiz:** [12-dataclasses-quiz.ipynb](12-dataclasses-quiz.ipynb)